# CNNs: hands-on (optional)
**LLMs From Basics · Lesson P6.1**

This notebook is optional practice for the lesson. Each exercise has:
1. a cell with a `# TODO` for you to fill in,
2. a **check** cell with `assert`s. Run it; a ✓ means success.

You'll count a conv layer's parameters, compute output sizes and receptive fields, then build a small CNN and train it on a **synthetic shapes dataset** that the notebook draws itself (no downloads). Everything runs on a CPU in about a minute (PyTorch is pre-installed on Colab).

Stuck? Solutions are at the very bottom.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)
torch.set_num_threads(2)
print("torch", torch.__version__)

## 1 · Count a conv layer's parameters
From the lesson: a conv layer with kernel $k \times k$, $C_{in}$ input channels and $C_{out}$ filters has

$$k \cdot k \cdot C_{in} \cdot C_{out} + C_{out}$$

parameters (the last term is one bias per filter). Write `conv_params(k, c_in, c_out, bias=True)`.

In [ ]:
def conv_params(k, c_in, c_out, bias=True):
    # TODO: return the number of parameters
    raise NotImplementedError("Exercise 1")

**Check 1.** Compare against PyTorch's own count, and against the dense layer from the lesson.

In [ ]:
for k, ci, co in [(3, 3, 64), (3, 64, 128), (5, 1, 6), (1, 256, 64)]:
    layer = nn.Conv2d(ci, co, k)
    assert conv_params(k, ci, co) == sum(p.numel() for p in layer.parameters()), (k, ci, co)
assert conv_params(3, 3, 64) == 1792
assert conv_params(5, 1, 6) == 156          # LeNet-5's first layer
assert conv_params(3, 64, 128, bias=False) == 73_728
dense = 224 * 224 * 3 * 1000 + 1000
print(f"3x3 conv, 3 -> 64 channels: {conv_params(3, 3, 64):,} params")
print(f"dense layer, 224x224x3 image -> 1000 units: {dense:,} params ({dense / conv_params(3, 3, 64):,.0f}x more)")
print("✓ exercise 1")

## 2 · Output size
The formula from P5.4: for input size $n$, kernel $k$, padding $p$ and stride $s$, the output size is $\lfloor (n + 2p - k)/s \rfloor + 1$. Write `conv_out(n, k, p=0, s=1)`; it works for pooling layers too (pooling has no padding by default).

In [ ]:
def conv_out(n, k, p=0, s=1):
    # TODO
    raise NotImplementedError("Exercise 2")

**Check 2**

In [ ]:
assert conv_out(32, 5) == 28                 # LeNet-5: 32x32 -> 28x28
assert conv_out(224, 3, p=1) == 224          # "same" padding
assert conv_out(224, 2, s=2) == 112          # 2x2 max pool
assert conv_out(224, 7, p=3, s=2) == 112     # ResNet's first layer
x = torch.zeros(1, 3, 57, 57)
for k, p, s in [(3, 1, 1), (3, 1, 2), (5, 0, 1), (4, 1, 3)]:
    y = nn.Conv2d(3, 8, k, padding=p, stride=s)(x)
    assert y.shape[-1] == conv_out(57, k, p, s), (k, p, s)
print("✓ exercise 2")

## 3 · Receptive field
Walk through the layers in order, starting from $r = 1$ and jump $j = 1$. For each layer with kernel $k$ and stride $s$:

$$r \leftarrow r + (k - 1)\cdot j, \qquad j \leftarrow j \cdot s$$

Write `receptive_field(layers)`, where `layers` is a list of `(k, s)` pairs, returning the final $r$.

In [ ]:
def receptive_field(layers):
    # TODO
    raise NotImplementedError("Exercise 3")

**Check 3.** The lesson's examples, plus VGG-16 (13 convs, 5 pools).

In [ ]:
assert receptive_field([(3, 1)] * 3) == 7
assert receptive_field([(3, 1), (2, 2), (3, 1), (3, 1)]) == 12
assert receptive_field([(5, 1), (5, 1)]) == 9
assert receptive_field([(3, 2), (3, 1), (3, 1)]) == 11
C, P = (3, 1), (2, 2)
vgg16 = [C, C, P, C, C, P, C, C, C, P, C, C, C, P, C, C, C]   # up to the last conv
assert receptive_field(vgg16) == 196
print("VGG-16 last conv layer sees", receptive_field(vgg16), "x", receptive_field(vgg16), "pixels")
print("✓ exercise 3")

## A synthetic shapes dataset
We draw 28×28 grayscale images of four shapes (circle, square, triangle, cross) at random sizes, positions and brightness, plus noise. A good test for a CNN: the same shape can appear anywhere, which is exactly what weight sharing handles well.

In [ ]:
CLASSES = ["circle", "square", "triangle", "cross"]

def make_shapes(n, size=28, seed=0):
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:size, 0:size] + 0.5
    X = np.zeros((n, 1, size, size), np.float32)
    y = rng.integers(0, 4, n)
    for i in range(n):
        r = rng.uniform(5, 9)
        cx, cy = rng.uniform(r + 1, size - r - 1, 2)
        dx, dy = xx - cx, yy - cy
        c = y[i]
        if c == 0:
            m = dx**2 + dy**2 <= r**2
        elif c == 1:
            s = 0.85 * r
            m = (abs(dx) <= s) & (abs(dy) <= s)
        elif c == 2:
            m = (dy <= 0.8 * r) & (dy >= -r + 2 * abs(dx))
        else:
            w = 0.3 * r
            m = ((abs(dx) <= w) & (abs(dy) <= r)) | ((abs(dy) <= w) & (abs(dx) <= r))
        X[i, 0] = m * rng.uniform(0.6, 1.0) + rng.normal(0, 0.15, (size, size))
    return torch.tensor(X), torch.tensor(y)

Xtr, ytr = make_shapes(4000, seed=0)
Xte, yte = make_shapes(800, seed=1)
fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, img, lab in zip(axes, Xtr[:8], ytr[:8]):
    ax.imshow(img[0], cmap="gray"); ax.set_title(CLASSES[lab]); ax.axis("off")
plt.show()

## 4 · Build a small CNN
Write `make_cnn()` returning an `nn.Sequential` with:

1. conv 3×3, 1 → 8 channels, padding 1, ReLU, max pool 2×2
2. conv 3×3, 8 → 16 channels, padding 1, ReLU, max pool 2×2
3. conv 3×3, 16 → 32 channels, padding 1, ReLU
4. **global average pooling** (`nn.AdaptiveAvgPool2d(1)`), `nn.Flatten()`, then `nn.Linear(32, 4)`

Shapes: 28×28 → 14×14 → 7×7 → one number per channel → 4 class scores.

In [ ]:
def make_cnn():
    # TODO
    raise NotImplementedError("Exercise 4")

**Check 4.** Shapes, parameter count (using your `conv_params`), and receptive field of the last conv layer.

In [ ]:
torch.manual_seed(0)
model = make_cnn()
assert model(torch.zeros(5, 1, 28, 28)).shape == (5, 4)
n_params = sum(p.numel() for p in model.parameters())
expected = conv_params(3, 1, 8) + conv_params(3, 8, 16) + conv_params(3, 16, 32) + 32 * 4 + 4
assert n_params == expected == 6020, n_params
rf = receptive_field([(3, 1), (2, 2), (3, 1), (2, 2), (3, 1)])
print(f"{n_params:,} parameters; the last conv layer sees {rf}x{rf} input pixels")
assert rf == 18
print("✓ exercise 4")

## 5 · Train it
Write `train(model, epochs)`: Adam with learning rate `3e-3`, mini-batches of 64 from a fresh random permutation each epoch, cross-entropy loss. Return the test accuracy (a float between 0 and 1) after the last epoch. The check expects **over 90%**.

In [ ]:
def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

def train(model, epochs=8, lr=3e-3, batch=64):
    # TODO: the usual loop from P4.2
    raise NotImplementedError("Exercise 5")

**Check 5** (about 5 to 20 seconds on a CPU)

In [ ]:
torch.manual_seed(0)
model = make_cnn()
t0 = time.time()
acc = train(model, epochs=8)
print(f"test accuracy {acc:.1%} in {time.time() - t0:.1f}s")
assert acc > 0.90, acc
print("✓ exercise 5")

## Experiment
1. **Look at the first-layer filters.** Run the cell below. With 3×3 kernels on noisy shapes they're small, but you'll often see edge-like patterns (positive on one side, negative on the other).
2. **Translation:** shift a test image a few pixels with `torch.roll` and check that the prediction usually doesn't change.
3. **An MLP for comparison:** `nn.Sequential(nn.Flatten(), nn.Linear(784, 64), nn.ReLU(), nn.Linear(64, 4))` has about 50,500 parameters, more than 8 times the CNN's 6,020. Train it with the same loop and compare accuracy.
4. Remove the two max-pool layers. What happens to the receptive field, the accuracy and the training time?

In [ ]:
w = model[0].weight.detach()[:, 0]          # 8 filters, 3x3
fig, axes = plt.subplots(1, 8, figsize=(12, 1.8))
for ax, f in zip(axes, w):
    ax.imshow(f, cmap="RdBu_r", vmin=-w.abs().max(), vmax=w.abs().max()); ax.axis("off")
plt.suptitle("first-layer filters (red = positive, blue = negative)")
plt.show()

img = Xte[:1]
for shift in [0, 2, -3]:
    pred = model(torch.roll(img, shifts=shift, dims=-1)).argmax(1).item()
    print(f"shifted by {shift:+d} px -> predicted {CLASSES[pred]} (true: {CLASSES[yte[0]]})")

---
# Solutions
Only peek after a real attempt.

### Exercise 1

In [ ]:
# Solution (copy into the exercise cell above)
def conv_params(k, c_in, c_out, bias=True):
    return k * k * c_in * c_out + (c_out if bias else 0)

### Exercise 2

In [ ]:
# Solution (copy into the exercise cell above)
def conv_out(n, k, p=0, s=1):
    return (n + 2 * p - k) // s + 1

### Exercise 3

In [ ]:
# Solution (copy into the exercise cell above)
def receptive_field(layers):
    r, j = 1, 1
    for k, s in layers:
        r += (k - 1) * j
        j *= s
    return r

### Exercise 4

In [ ]:
# Solution (copy into the exercise cell above)
def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(32, 4),
    )

### Exercise 5

In [ ]:
# Solution (copy into the exercise cell above)
def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(1) == y).float().mean().item()

def train(model, epochs=8, lr=3e-3, batch=64):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        perm = torch.randperm(len(Xtr))
        for b in range(0, len(Xtr), batch):
            idx = perm[b:b + batch]
            loss = F.cross_entropy(model(Xtr[idx]), ytr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
        print(f"epoch {ep + 1}: last batch loss {loss.item():.3f}, test acc {accuracy(model, Xte, yte):.3f}")
    return accuracy(model, Xte, yte)